# Capítulo 2 · Modelación de desplazamientos verticales: del sismo a la condición inicial del tsunami

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/modelacion-sismica-tsunami-espam/blob/main/cuadernos/Cap2_deformacion_cosismica.ipynb)

**Modelación Sísmica y Tsunami (IRD-505) · Ingeniería en Riesgos de Desastres · ESPAM MFL**
Docente: Cristian Paliz Acosta, MSc.

En la Unidad 1 describieron la fuente de su sismo con una **ficha**: un rectángulo con deslizamiento uniforme. En la práctica de consolidación ese rectángulo se convirtió, con Okada (1985), en un mapa de cuánto sube y baja el terreno. Ese modelo es útil, pero simplifica mucho: la falla real no es plana, el deslizamiento no es igual en toda ella, la ruptura no ocurre en un instante y el mar no copia exactamente lo que hace el fondo.

En este cuaderno la fuente se vuelve realista **capa por capa**. Cada capa agrega un fenómeno, lo cuantifica y lo compara con la anterior. Al final queda guardada la **deformación del fondo marino** con la que se modelará la propagación del tsunami (Unidad 3) y la inundación (Unidad 4).

**Ruta del capítulo**

| Paso | Qué hacen | Tema de la Unidad 2 | Producto |
|---|---|---|---|
| P | Predicen **antes** de calcular | Razonamiento físico | Predicción registrada |
| 0 | Conocen los datos del evento | Tensor de momento y falla finita del USGS | Mapa de deslizamiento |
| 1 | Miran la falla en profundidad | Geometría de la interfase de subducción | Corte por su localidad |
| 2 | Comparan uniforme y falla finita | Superposición de subfallas | Mapas, perfil y tabla de métricas |
| 3 | Cambian la rigidez de la roca somera | Rigidez, deslizamiento y sismos tsunami | Perfil con μ variable |
| 4 | Animan la ruptura | Cinemática y propagación de la ruptura | GIF y figura de 4 instantes |
| 5 | Pasan del fondo a la superficie del mar | Aporte horizontal y filtro de Kajiura | Mapas de uz, uh y η0 |
| 6 | Validan con observaciones | GPS, InSAR y cambios en la costa | RMSE y gráfico 1:1 |
| 7 | Analizan la sensibilidad | Incertidumbre de los parámetros | Gráfico de tornado |
| 8 | Comparan predicción y modelo | Aprendizaje | Tabla predicción vs. modelo |
| 9 | Exportan para la Unidad 3 | Condición inicial del tsunami | Paquete `modelo_U2` |

**Cómo usar el cuaderno.** Hagan una copia (*Archivo → Guardar una copia en Drive*), ejecuten las celdas **en orden** (Mayúsculas + Enter) y edítenlas solo donde aparece ✏️. Cada paso termina con preguntas 📝 que se responden en el Capítulo 2 del informe. Los datos de los 11 eventos ya están en el repositorio del curso; si faltara alguno, el cuaderno lo descarga.

**Convenciones** (las mismas de la Unidad 1): uz > 0 es levantamiento; en los perfiles la distancia es negativa hacia el mar (fosa) y positiva hacia tierra; rumbo, buzamiento y rake con la regla de la mano derecha.

## Preparación del entorno
Descarga el repositorio del curso (código del paquete `deformacion` y datos de los eventos) e importa las librerías. No hace falta editar. Tarda menos de un minuto.

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
REPO_URL = "https://github.com/crispaliz/modelacion-sismica-tsunami-espam"
CARPETA = "/content/modelacion-sismica-tsunami-espam" if os.path.exists("/content") else os.path.abspath("..")
if not os.path.exists(os.path.join(CARPETA, "src", "deformacion")):
    !git clone -q --depth 1 {REPO_URL} {CARPETA}
sys.path.insert(0, os.path.join(CARPETA, "src"))
RAIZ_DATOS = os.path.join(CARPETA, "datos", "eventos")

import numpy as np
import matplotlib.pyplot as plt
import deformacion as dfm
from deformacion import datos, graficos as gr
from deformacion.exportar import escribir_dtopo_tt3, escribir_topo_tt3, guardar_campos
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
print("Paquete deformacion", dfm.__version__, "· datos en", RAIZ_DATOS)

## Datos del grupo ✏️
Elijan su **evento** y escriban su **localidad costera** (la misma del Capítulo 1). En Colab el menú aparece a la derecha de la celda. Si su localidad no está en la lista del evento, escriban su nombre y sus coordenadas en `LAT_OTRA` y `LON_OTRA` (grados decimales, sur y oeste negativos).

> Desde este capítulo cada grupo trabaja **solo con su evento y su localidad**; Pedernales 2016 fue el caso comparativo del Capítulo 1.

In [ ]:
#@title ✏️ Datos del grupo
GRUPO = "Grupo X"                 #@param {type:"string"}
EVENTO = "Pedernales 2016"        #@param ["Pedernales 2016", "Maule 2010", "Illapel 2015", "Iquique 2014", "Pisco 2007", "Camaná 2001", "Tohoku 2011", "Nicaragua 1992", "Mentawai 2010", "Samoa 2009", "Océano Índico 2012"]
LOCALIDAD = "Pedernales"          #@param {type:"string"}
LAT_OTRA = 0.0                    #@param {type:"number"}
LON_OTRA = 0.0                    #@param {type:"number"}

if LOCALIDAD in dfm.LOCALIDADES[EVENTO]:
    LOC_LAT, LOC_LON = dfm.LOCALIDADES[EVENTO][LOCALIDAD]
else:
    LOC_LAT, LOC_LON = LAT_OTRA, LON_OTRA
    print("⚠️ Localidad fuera de la lista: se usan LAT_OTRA y LON_OTRA. Opciones:",
          list(dfm.LOCALIDADES[EVENTO]))
print(f"{GRUPO} · {EVENTO} ({dfm.EVENTOS[EVENTO]['tipo']}) · {LOCALIDAD} ({LOC_LAT:.3f}, {LOC_LON:.3f})")
print("\nPregunta guía del capítulo 2:\n ", dfm.PREGUNTA_U2[EVENTO])

## Paso P ✏️ · Predicción antes de calcular

Sin ejecutar nada más, con su ficha del Capítulo 1 y lo visto en clase, completen la celda siguiente. **No se corrige la predicción: se evalúa que exista y que esté razonada.** En el Paso 8 la compararán con el modelo.

Pistas para razonar:
- Si el deslizamiento real se concentra en un parche (no es uniforme), ¿el levantamiento máximo será mayor o menor que el del rectángulo?
- ¿Qué parte de la deformación caerá en el mar? Piensen dónde está la fosa y dónde la costa.
- Una ruptura avanza a unos 2–3 km/s. ¿Cuánto tardará en romper su falla de largo L?
- El mar tiene kilómetros de profundidad. ¿La ola inicial será igual, mayor o menor que la deformación del fondo?

In [ ]:
# ✏️ Completen ANTES de ejecutar el resto del cuaderno
PREDICCION = dict(
    uz_max_falla_finita_m = 1.0,      # levantamiento máximo esperado con la falla finita (m)
    mayor_que_uniforme    = "sí",     # ¿será mayor que el del rectángulo uniforme? "sí" / "no"
    fraccion_en_mar       = 0.8,      # fracción del levantamiento que cae en el mar (0 a 1)
    localidad             = "baja",   # su localidad: "sube" / "baja"
    duracion_ruptura_s    = 60,       # duración de la ruptura (s)
    eta0_frente_al_fondo  = "menor",  # η0 máximo frente al uz máximo del fondo: "mayor" / "igual" / "menor"
    razonamiento = "Escriban aquí, en 3 a 5 líneas, por qué esperan esos valores.",
)
print("Predicción registrada:", PREDICCION)

## 🧭 Habilidad transversal · Resolución de problemas complejos

En este capítulo no hay una respuesta única. El problema es **complejo** por cuatro razones:
- la información es **incompleta**: no todos los eventos tienen falla finita ni observaciones;
- los modelos **no son únicos**: otra falla finita daría otro resultado;
- los **supuestos cambian el resultado**: rigidez, geometría, tipo de fuente;
- hay que **decidir** con incertidumbre qué fuente se entrega a la Unidad 3.

La habilidad se trabaja en cuatro fases. Cada una tiene un lugar en el cuaderno:

| Fase | Qué hacen | Dónde |
|---|---|---|
| 1. Comprender | Identifican qué se sabe, qué falta y qué se pregunta | Pregunta guía, Paso P, Paso 0 |
| 2. Representar | Eligen cómo modelar el problema y con qué supuestos | Pasos 1, 2 y 3 |
| 3. Decidir y ejecutar | Comparan alternativas y eligen con evidencia | **Bitácora de decisiones** (Pasos 3, 5 y 6) |
| 4. Monitorear y reflexionar | Detectan dónde falla su solución, prueban hipótesis y revisan su proceso | **Diagnóstico** (Paso 6) y **cierre metacognitivo** (Paso 8) |

**La bitácora de decisiones.** Cada vez que el cuaderno les pide decidir, registran qué **alternativas** consideraron, qué **eligieron**, con qué **evidencia** (una figura, una cifra, una referencia) y qué **riesgo** asumen si se equivocan. No se califica que la decisión sea «la correcta», sino que esté **justificada con evidencia** y que reconozca sus límites. La bitácora se exporta con el paquete del Paso 9 y es un criterio de la rúbrica del Capítulo 2 (ver `guias/Capitulo2_guia_y_rubrica.md`).

La celda siguiente prepara la bitácora. No hace falta editarla.

In [ ]:
BITACORA, DIAGNOSTICO, METACOGNICION = [], {}, {}
_PLANTILLA = "..."

def decision(paso, pregunta, alternativas, eleccion, evidencia, riesgo):
    """Registra (o reemplaza) la decisión de un paso en la bitácora."""
    global BITACORA
    BITACORA = [d for d in BITACORA if d["paso"] != paso]
    d = dict(paso=paso, pregunta=pregunta, alternativas=alternativas, eleccion=eleccion,
             evidencia=evidencia, riesgo=riesgo)
    BITACORA.append(d)
    vacios = [k for k, v in d.items() if not str(v).strip() or str(v).strip() == _PLANTILLA]
    print(f"Decisión del paso {paso} registrada." + (f" ⚠️ Faltan: {', '.join(vacios)}" if vacios else " ✅"))

def mostrar_bitacora():
    import pandas as pd
    return pd.DataFrame(BITACORA).set_index("paso") if BITACORA else "La bitácora está vacía."

print("Bitácora lista.")

---
## Paso 0 · Qué sabemos del evento

Se cargan los datos que dejó preparados el docente en `datos/eventos/<evento>/`:

- **Origen** (hipocentro): dónde y cuándo **empezó** la ruptura.
- **Tensor de momento W-phase** del USGS: el tamaño (M0, Mw), el **centroide** (el «centro de gravedad» del deslizamiento) y los dos **planos nodales**. Es la información que usaron en la ficha del Capítulo 1.
- **Modelo de falla finita** del USGS: la falla dividida en decenas o cientos de **subfallas**, cada una con su deslizamiento, su rake y el instante en que empezó a deslizar. Se obtiene **invirtiendo** registros sísmicos (y a veces GPS e InSAR): se busca la distribución de deslizamiento que mejor explica las ondas registradas.

> ⚠️ **Un modelo de falla finita es un modelo, no una observación.** Otro grupo de investigación, con otros datos o con otra geometría, obtiene una distribución distinta. Úsenlo como la mejor estimación disponible y citen siempre su origen (USGS, identificador del evento).

**Cómo leer el mapa:** el color de cada subfalla es su deslizamiento; la estrella amarilla es el epicentro (donde empezó), el rombo negro el centroide (donde se concentró el momento) y el círculo rojo su localidad. Que el epicentro y el parche de mayor deslizamiento no coincidan es lo normal: la ruptura nace en un punto y crece.

In [ ]:
try:
    D = datos.cargar_evento(EVENTO, RAIZ_DATOS)
except FileNotFoundError:
    print("Los datos no están en el repositorio: se descargan ahora (requiere internet, 1–3 min).")
    datos.preparar_evento(EVENTO, RAIZ_DATOS)
    D = datos.cargar_evento(EVENTO, RAIZ_DATOS)

META = D["meta"]; FF = D["falla_finita"]
BLON, BLAT, BZ = D["batimetria"]
t = META.get("tensor", {})
print(f"USGS {META['id_usgs']} · {META['titulo']}")
print(f"Origen {META['tiempo_utc']} · hipocentro {META['hipocentro']}")
if t:
    print(f"Tensor ({t.get('fuente')}): M0 = {t['M0_Nm']:.3e} N·m → Mw {dfm.momento_a_mw(t['M0_Nm']):.2f}; "
          f"centroide a {t['centroide']['prof_km']} km")
    print(f"  Plano 1: rumbo {t['plano1']['rumbo']}°, buz {t['plano1']['buz']}°, rake {t['plano1']['rake']}°")
    print(f"  Plano 2: rumbo {t['plano2']['rumbo']}°, buz {t['plano2']['buz']}°, rake {t['plano2']['rake']}°")
if FF is not None:
    r = FF.resumen()
    print(f"\nFalla finita: {r['n_subfallas']} subfallas · slip máx {r['slip_max_m']:.2f} m · "
          f"medio {r['slip_medio_m']:.2f} m · de {r['prof_min_km']:.1f} a {r['prof_max_km']:.1f} km · "
          f"Mw {r['Mw']:.2f} (con μ = 30 GPa)")
else:
    print("\nEste evento no tiene falla finita del USGS: se trabajará con el rectángulo uniforme "
          "(capa 2) subdividido para la cinemática (capa 4).")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7))
gr.mapa_base(ax, BLON, BLAT, BZ)
if FF is not None:
    pc = gr.dibujar_falla(ax, FF)
    plt.colorbar(pc, ax=ax, shrink=0.7, label="deslizamiento (m)")
h = META["hipocentro"]
gr.marcar(ax, h["lon"], h["lat"], "epicentro", "*", "gold", 16)
if t:
    gr.marcar(ax, t["centroide"]["lon"], t["centroide"]["lat"], "centroide", "D", "k", 7)
gr.marcar(ax, LOC_LON, LOC_LAT, LOCALIDAD, "o", "crimson", 8)
ax.set_title(f"{EVENTO}: falla finita sobre la batimetría")
plt.show()

📝 **Preguntas (Paso 0)**
1. ¿Dónde está el parche de mayor deslizamiento respecto al epicentro y a su localidad? ¿Bajo el mar o bajo tierra?
2. ¿Cuál es el deslizamiento máximo y el medio? Compárenlos con el D de su ficha del Capítulo 1.
3. El Mw que imprime el cuaderno para la falla finita se calcula con μ = 30 GPa. Si difiere del Mw del tensor, ¿qué explica la diferencia? (Pista: el USGS usa una rigidez que cambia con la profundidad.)
4. Si su evento **no** tiene falla finita, ¿qué consecuencias tiene para su modelo?

---
## Paso 1 · Geometría: la falla en profundidad

En la ficha la falla era un rectángulo **plano**. En una zona de subducción la interfase entre las placas se **curva**: cerca de la fosa es casi horizontal (5°–10°) y se empina con la profundidad (20°–30° a 40–50 km). Esto importa porque el buzamiento controla cuánto del deslizamiento se convierte en movimiento vertical (recuerden D · sen δ y la Unidad 1).

El corte se traza **perpendicular al rumbo** y pasa por su localidad:
- **Relleno beige y línea negra:** el terreno y el fondo marino (ETOPO 2022); el celeste es el mar.
- **Círculos:** centroides de las subfallas de la falla finita, coloreados y escalados por su deslizamiento.
- **Línea discontinua verde** (si aparece): techo de la placa según **Slab2** (Hayes et al., 2018), el modelo de geometría de las zonas de subducción del USGS.
- **Línea roja:** su localidad.

> ⚠️ La escala vertical está **exagerada**: 80 km de profundidad ocupan casi lo mismo que 450 km de distancia. Los ángulos en la figura se ven más empinados que en la realidad.

**Error frecuente:** si la localidad está lejos, a lo largo del rumbo, del parche de deslizamiento, el corte no pasa por la parte importante de la ruptura. Lo notarán porque las subfallas quedan proyectadas pero el perfil de deformación del Paso 2 sale débil.

In [ ]:
RUMBO = dfm.rumbo_medio(FF) if FF is not None else dfm.elegir_plano(t, dfm.EVENTOS[EVENTO]["plano"])[0]["rumbo"]
S, PLON, PLAT = dfm.perfil(LOC_LON, LOC_LAT, RUMBO, largo_mar=300, largo_tierra=150)
zp = dfm.interpolar_malla(BLON, BLAT, BZ, PLON, PLAT)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.fill_between(S, zp / 1000, -80, color="#E4DAC4"); ax.fill_between(S, 0, zp / 1000, where=zp < 0, color="#BFD9E6")
ax.plot(S, zp / 1000, "k", lw=1)
def distancia_al_corte(lon, lat):
    # distancia (km) en la dirección del perfil: + hacia tierra, − hacia el mar
    e, n = dfm.geo_a_km(lon, lat, LOC_LON, LOC_LAT)
    phi = np.radians(RUMBO)
    return e * np.cos(phi) - n * np.sin(phi)
if FF is not None:
    sx = np.array([distancia_al_corte(s.lon, s.lat) for s in FF.subfallas])
    sz = np.array([s.prof for s in FF.subfallas]); ss = FF.slip
    sc = ax.scatter(sx, -sz, c=ss, s=10 + 25 * ss / ss.max(), cmap="YlOrRd", edgecolor="0.3", lw=0.3)
    plt.colorbar(sc, ax=ax, label="deslizamiento (m)")
if D["slab2"] is not None:
    SL = dfm.interpolar_malla(*D["slab2"], PLON, PLAT)
    ax.plot(S, -SL, "--", color="#1C6E7D", lw=2, label="Slab2 (techo de la placa)")
    ax.legend(loc="lower left")
ax.axvline(0, color="crimson", lw=1); ax.text(2, 2, LOCALIDAD, color="crimson")
ax.set_ylim(-80, 5); ax.set_xlabel("distancia perpendicular al rumbo (km) · negativo = hacia el mar")
ax.set_ylabel("elevación / profundidad (km)"); ax.set_title(f"Corte por {LOCALIDAD} (rumbo {RUMBO:.0f}°) · escala vertical exagerada")
plt.show()

📝 **Preguntas (Paso 1)**
1. ¿A qué distancia de su localidad está la fosa (la parte más profunda del corte)? ¿Y el deslizamiento máximo?
2. ¿Las subfallas siguen una línea recta o se curvan? Estimen el buzamiento entre 0 y 20 km y entre 20 y 40 km de profundidad (Δz / Δx).
3. ¿El buzamiento de su ficha representa bien la parte de la falla que más deslizó?
4. Si aparece Slab2, ¿la falla finita del USGS coincide con el techo de la placa? Si no aparece, describan la interfase con la falla finita.

---
## Paso 2 · Rectángulo uniforme frente a falla finita

**La idea física.** El medio de Okada es **elástico lineal**: el desplazamiento producido por dos fallas es la **suma** de los desplazamientos de cada una. Por eso una falla finita se modela así:

$$u_z(x, y) = \sum_{i=1}^{N} u_z^{\text{Okada}}\left(x, y;\ \text{subfalla}_i\right)$$

Cada subfalla es un pequeño rectángulo con su propio deslizamiento, rake, rumbo y buzamiento. El rectángulo uniforme del Capítulo 1 es el caso N = 1.

**¿Qué cambia?** Lejos de la falla los dos modelos se parecen: a gran distancia solo «se ve» el momento total. **Cerca** (justo donde están el fondo marino que genera el tsunami y su costa) la distribución del deslizamiento manda: un parche concentrado produce un levantamiento más alto y más angosto que el mismo momento repartido uniformemente.

### ✏️ Su ficha del Capítulo 1
**Recomendado:** suban el CSV de la ficha que exportó el cuaderno `Cap1_fuente_sismica.ipynb` (ícono de carpeta, a la izquierda) y escriban su nombre en `FICHA_CSV`. Se completan solas la ficha y la localidad.

Sin CSV, completen `FICHA` a mano. Lo que quede en `None` se toma del **tensor de momento del USGS** (plano de menor buzamiento para interfase) y de las **leyes de escala**: Strasser et al. (2010) para interfase y Wells y Coppersmith (1994) para los otros tipos.

`PASO` es la separación de la malla en grados: 1/30 equivale a 2 minutos de arco, unos 3,7 km. Mallas más finas tardan más.

In [ ]:
#@title ✏️ Ficha de la fuente (Capítulo 1)
FICHA_CSV = ""     #@param {type:"string"}
FICHA = dict(
    lon=None, lat=None, prof_centroide=None,   # grados y km (centroide)
    rumbo=None, buz=None, rake=None,           # grados
    L=None, W=None,                            # km
    slip=None,                                 # m (si None: M0 / (μ L W))
    M0=None,                                   # N·m (si None: el del tensor del USGS)
    mu=30e9,                                   # Pa
)
PASO = 1/30   #@param {type:"number"}
NU = 0.25

if FICHA_CSV:
    import pandas as pd
    _f = pd.read_csv(FICHA_CSV); _f = _f[_f["evento"] == EVENTO]
    if _f.empty:
        raise ValueError(f"El CSV no tiene la ficha de {EVENTO}")
    _v = dict(zip(_f["parametro"], _f["valor"].astype(float)))
    FICHA.update(lon=_v["lon"], lat=_v["lat"], rumbo=_v["rumbo"], buz=_v["buz"], rake=_v["rake"],
                 L=_v["L"], W=_v["W"], slip=_v.get("D"), M0=_v.get("M0"), mu=_v.get("mu", 30) * 1e9,
                 prof_centroide=_v["z_sup"] + _v["W"] / 2 * np.sin(np.radians(_v["buz"])))
    if "loc_lat" in _v:
        LOC_LAT, LOC_LON = _v["loc_lat"], _v["loc_lon"]
        LOCALIDAD = str(_f.loc[_f["parametro"] == "loc_lat", "metodo"].iloc[0])
    print("Ficha leída del CSV:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in FICHA.items()})
    print("Localidad:", LOCALIDAD, LOC_LAT, LOC_LON)

In [ ]:
GEOM = ("lon", "lat", "prof_centroide", "rumbo", "buz", "rake", "L", "W")
if all(FICHA[k] is not None for k in GEOM):
    FU = dfm.falla_uniforme(FICHA["lon"], FICHA["lat"], FICHA["prof_centroide"], FICHA["rumbo"], FICHA["buz"],
                            FICHA["rake"], M0=FICHA["M0"] or t["M0_Nm"], L=FICHA["L"], W=FICHA["W"],
                            slip=FICHA["slip"], mu=FICHA["mu"], nombre=EVENTO)
    FU.origen = "ficha del Capítulo 1"
    h = META["hipocentro"]; FU.hipocentro = (h["lon"], h["lat"], h["prof_km"])
else:
    FU = dfm.falla_desde_tensor(EVENTO, META, mu=FICHA["mu"], L=FICHA["L"], W=FICHA["W"],
                                slip=FICHA["slip"], prof=FICHA["prof_centroide"])
s0 = FU.subfallas[0]
print(f"Rectángulo uniforme ({FU.origen}):")
print(f"  rumbo {s0.rumbo:.0f}° · buz {s0.buz:.0f}° · rake {s0.rake:.0f}° · L = {s0.L:.0f} km · W = {s0.W:.0f} km · "
      f"slip = {s0.slip:.2f} m · borde sup {s0.prof_sup:.1f} km · Mw {FU.mw(FICHA['mu']):.2f}")

ext = FF.extension(1.5) if FF is not None else FU.extension(2.0)
ext = (min(ext[0], LOC_LON - 0.5), max(ext[1], LOC_LON + 0.5), min(ext[2], LOC_LAT - 0.5), max(ext[3], LOC_LAT + 0.5))
REG = (max(ext[0], BLON.min()), min(ext[1], BLON.max()), max(ext[2], BLAT.min()), min(ext[3], BLAT.max()))
LON, LAT = dfm.malla(REG, PASO)
Z = dfm.interpolar_malla(BLON, BLAT, BZ, LON, LAT)
print("Malla:", LON.shape, "puntos")

import time; t0 = time.time()
U_UNI = FU.desplazamiento(LON, LAT, NU)
U_FF = FF.desplazamiento(LON, LAT, NU) if FF is not None else None
print(f"Okada listo en {time.time() - t0:.1f} s")

In [ ]:
vmax = max(np.abs(U_UNI[2]).max(), np.abs(U_FF[2]).max() if U_FF is not None else 0)
n = 2 if U_FF is not None else 1
fig, axs = plt.subplots(1, n, figsize=(6.5 * n, 6), squeeze=False)
gr.mapa_campo(axs[0, 0], LON, LAT, U_UNI[2], Z, "uz · rectángulo uniforme", vmax)
gr.contorno_falla(axs[0, 0], FU)
if U_FF is not None:
    gr.mapa_campo(axs[0, 1], LON, LAT, U_FF[2], Z, "uz · falla finita USGS", vmax)
    gr.contorno_falla(axs[0, 1], FF, color="0.4")
for a in axs[0]:
    gr.marcar(a, LOC_LON, LOC_LAT, LOCALIDAD, "o", "crimson", 7)
plt.tight_layout(); plt.show()

S2, P2LON, P2LAT = dfm.perfil(LOC_LON, LOC_LAT, RUMBO, 300, 150)
fig, ax = plt.subplots(figsize=(10, 3.8))
for nombre_f, F, col in (("uniforme", FU, "#1C6E7D"), ("falla finita", FF, "#B04616")):
    if F is None: continue
    uzp = F.desplazamiento(P2LON, P2LAT, NU)[2]
    ax.plot(S2, uzp, color=col, lw=2, label=nombre_f)
    for c0 in dfm.cruces_de_cero(S2, uzp):
        ax.axvline(c0, color=col, ls=":", lw=1)
zc = dfm.interpolar_malla(BLON, BLAT, BZ, P2LON, P2LAT)
costa = dfm.cruces_de_cero(S2, zc)
for c0 in costa: ax.axvline(c0, color="k", lw=1)
ax.axhline(0, color="0.3", lw=0.8); ax.axvline(0, color="crimson", lw=1)
ax.set_xlabel("km (negativo = mar) · línea negra: costa · punteadas: línea de cero"); ax.set_ylabel("uz (m)")
ax.legend(); ax.set_title(f"Perfil perpendicular al rumbo por {LOCALIDAD}"); plt.show()

import pandas as pd
filas = {"uniforme": dfm.metricas(LON, LAT, U_UNI[2], Z)}
if U_FF is not None: filas["falla finita"] = dfm.metricas(LON, LAT, U_FF[2], Z)
TABLA2 = pd.DataFrame(filas).T
TABLA2[["uz_max_m", "uz_min_m", "uz_max_mar_m", "uz_min_tierra_m", "fraccion_levantamiento_en_mar"]].round(3)

**Cómo leer las figuras**
- **Mapas:** rojo sube, azul baja, con la misma escala de color en los dos para poder compararlos. La línea negra continua es la costa; la discontinua, la **línea de cero** (donde se pasa de levantamiento a hundimiento).
- **Perfil:** los dos modelos por su localidad. Las líneas punteadas verticales marcan la línea de cero de cada modelo; la línea negra, la costa.
- **Tabla:** `uz_max_mar_m` es el levantamiento máximo **sobre el mar** (el que genera el tsunami); `uz_min_tierra_m` el mayor hundimiento **en tierra** (el que agrava la inundación); `fraccion_levantamiento_en_mar` cuánto del volumen levantado cae en el mar.

> Si en el perfil de la falla finita aparecen pequeños escalones, son los bordes de subfallas someras con deslizamiento distinto: un efecto de dividir la falla en rectángulos, no un fenómeno físico.

📝 **Preguntas (Paso 2)**
1. ¿Cuánto cambian el levantamiento máximo, el hundimiento máximo y la posición de la línea de cero entre ambos modelos?
2. ¿Qué fracción del levantamiento cae en el mar en cada caso? ¿Por qué esa cifra importa más para el tsunami que el máximo absoluto?
3. ¿Su localidad sube o baja? ¿Cuánto? ¿Qué implica para la inundación, el drenaje y los muelles?
4. ¿En qué zona los dos modelos se parecen y en cuál difieren? Relaciónenlo con la distancia a la falla.

---
## Paso 3 · Rigidez: cuando la roca somera es más blanda

**La idea física.** El momento sísmico es

$$M_0 = \mu \, A \, D$$

donde μ es la rigidez (Pa), A el área de ruptura y D el deslizamiento medio. Las ondas sísmicas miden **M0**, no D. Para pasar de M0 a D hay que suponer μ. Con el **mismo M0**, si la roca es más blanda (μ menor), el deslizamiento es **mayor**, y con él el levantamiento.

Cerca de la fosa la falla corta sedimentos y roca fracturada, mucho menos rígidos que la corteza profunda. Bilek y Lay (1999) mostraron que la rigidez a lo largo de las megafallas de subducción aumenta con la profundidad. Es una de las explicaciones de los **sismos tsunami** (Nicaragua 1992, Mentawai 2010): sismos de magnitud moderada, ruptura lenta y somera, y tsunamis mucho mayores de lo esperado.

**Qué hace la celda.** Se supone una rigidez que crece con la profundidad: `MU_SOMERO` hasta `Z1` km, una rampa lineal hasta `MU_PROFUNDO` en `Z2` km. A cada subfalla se le conserva su momento (calculado con 30 GPa) y se recalcula su deslizamiento:

$$D_{\text{nuevo}} = D \cdot \frac{30\ \text{GPa}}{\mu(z)}$$

> ⚠️ La rampa es **ilustrativa**: sirve para entender el efecto, no es un modelo de rigidez de su zona. Los valores son ajustables: justifiquen en el informe los que adopten. Si su evento no tiene falla finita, se usa el rectángulo uniforme subdividido.

**Cómo leer la figura:** a la izquierda, la rigidez supuesta frente a la profundidad (la línea discontinua es la referencia de 30 GPa). A la derecha, el perfil con rigidez constante y con rigidez variable, con el mismo momento.

In [ ]:
#@title ✏️ Rigidez supuesta
MU_SOMERO = 10e9    #@param {type:"number"}
MU_PROFUNDO = 40e9  #@param {type:"number"}
Z1 = 5.0            #@param {type:"number"}
Z2 = 20.0           #@param {type:"number"}
kw = dict(mu_somero=MU_SOMERO, mu_profundo=MU_PROFUNDO, z1=Z1, z2=Z2)

zz = np.linspace(0, 60, 121)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].plot(dfm.mu_rampa(zz, **kw) / 1e9, zz); axs[0].invert_yaxis()
axs[0].axvline(30, color="0.5", ls="--", label="μ de referencia 30 GPa"); axs[0].legend()
axs[0].set_xlabel("μ (GPa)"); axs[0].set_ylabel("profundidad (km)"); axs[0].set_title("Rigidez supuesta")

BASE = FF if FF is not None else dfm.subdividir(FU, 12, 6, FU.hipocentro)
BASE_MU = dfm.reescalar_por_rigidez(BASE, **kw)
u_b = BASE.desplazamiento(P2LON, P2LAT, NU)[2]
u_m = BASE_MU.desplazamiento(P2LON, P2LAT, NU)[2]
axs[1].plot(S2, u_b, color="#B04616", lw=2, label="μ = 30 GPa")
axs[1].plot(S2, u_m, color="#12263A", lw=2, ls="--", label="μ(z), mismo M0")
axs[1].axhline(0, color="0.3", lw=0.8); axs[1].legend(); axs[1].set_xlabel("km (negativo = mar)")
axs[1].set_ylabel("uz (m)"); axs[1].set_title("Mismo momento, distinta rigidez")
plt.tight_layout(); plt.show()
print(f"Slip máximo: {BASE.slip.max():.2f} m → {BASE_MU.slip.max():.2f} m · "
      f"uz máximo en el perfil: {u_b.max():.2f} m → {u_m.max():.2f} m")

### 🧭 Bitácora · Decisión 1 ✏️ · ¿Qué rigidez adoptan para la parte somera de la falla?
Miren el perfil anterior y la literatura de su evento. Consideren al menos **dos alternativas** (por ejemplo, μ constante de 30 GPa y la rampa que probaron). La evidencia debe ser concreta: una cifra del perfil, una característica del evento (¿sismo tsunami?, ¿sedimentos en la fosa?) o una referencia.

In [ ]:
#@title ✏️ Decisión 1 · Rigidez
decision(
    paso="3",
    pregunta="¿Qué rigidez adopto para la parte somera de la falla?",
    alternativas="...",   # p. ej.: "μ = 30 GPa constante | rampa 10→40 GPa entre 5 y 20 km"
    eleccion="...",       # cuál eligen y con qué valores
    evidencia="...",      # cifra o figura de este cuaderno + argumento de la literatura (con cita)
    riesgo="...",         # qué pasa con el tsunami modelado si su elección está equivocada
)

📝 **Preguntas (Paso 3)**
1. ¿En qué porcentaje cambia el levantamiento máximo? ¿Dónde se concentra el cambio: cerca de la fosa o en profundidad? ¿Por qué?
2. Si el deslizamiento máximo de su evento está profundo (más allá de `Z2`), el efecto puede incluso invertirse. ¿Ocurre en su caso?
3. ¿Es razonable suponer roca somera blanda en su evento? Busquen un argumento en la literatura (tipo de sismo, sedimentos en la fosa, duración de la ruptura).

---
## Paso 4 · Cinemática: la ruptura avanza y el fondo se deforma

**La idea física.** Un sismo no ocurre en un instante: la ruptura nace en el **hipocentro** y se propaga por la falla a unos 2–3 km/s (la **velocidad de ruptura**). Cada punto de la falla empieza a deslizar cuando le llega el **frente de ruptura** y completa su deslizamiento en unos segundos (el **tiempo de subida**).

El modelo de falla finita del USGS trae para cada subfalla su tiempo de inicio (`t_rup`) y su duración. Como el medio es lineal, la deformación en el instante t es

$$u_z(x, y, t) = \sum_i u_z^{(i)}(x, y)\; S\!\left(\frac{t - t_{\text{rup},i}}{T_i}\right)$$

donde S va de 0 a 1 de forma suave (función coseno). Si no hay falla finita, el rectángulo se divide en 16 × 8 subfallas y la ruptura sale del hipocentro con velocidad `VR`.

**Cómo leer las figuras**
- **Cuatro instantes:** la deformación acumulada hasta ese momento. Los cuadros grises son las subfallas que **están deslizando** en ese instante: el frente de ruptura.
- **Animación (GIF):** la misma secuencia completa. Se guarda para el informe y para la presentación.
- **Series de tiempo:** cómo se mueve el suelo en su localidad y en el punto de máximo levantamiento.

`PASO_ANIM` es la malla de la animación (1/15° ≈ 7 km), más gruesa que la del Paso 2 para que el cálculo sea rápido.

In [ ]:
#@title ✏️ Cinemática de la ruptura
VR = 2.5            #@param {type:"number"}
PASO_ANIM = 1/15    #@param {type:"number"}
FK = FF if FF is not None else dfm.subdividir(FU, 16, 8, FU.hipocentro, vr=VR)
ALON, ALAT = dfm.malla(REG, PASO_ANIM)
AZ = dfm.interpolar_malla(BLON, BLAT, BZ, ALON, ALAT)
CAMPOS = FK.desplazamiento_por_subfalla(ALON, ALAT, NU, ("uz",))["uz"]
TIEMPOS = dfm.tiempos_sugeridos(FK, 36)
UZT = dfm.campo_en_el_tiempo(CAMPOS, FK, TIEMPOS)
print(f"Duración de la ruptura ≈ {TIEMPOS[-1]:.0f} s · {len(TIEMPOS)} cuadros · malla {ALON.shape}")

fig, axs = plt.subplots(1, 4, figsize=(16, 4.4))
vmax_t = np.abs(UZT[-1]).max()
for a, k in zip(axs, np.linspace(len(TIEMPOS) * 0.15, len(TIEMPOS) - 1, 4).astype(int)):
    gr.mapa_campo(a, ALON, ALAT, UZT[k], AZ, f"t = {TIEMPOS[k]:.0f} s", vmax_t, barra=False)
    act, hecha = dfm.frente_de_ruptura(FK, TIEMPOS[k])
    gr.contorno_falla(a, FK, color="0.6")
    gr.dibujar_falla(a, FK, campo=np.ones(len(FK)), cmap="Greys", vmax=1.5, alpha=0.45, solo=act)
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib import animation
from IPython.display import Image, display
fig, ax = plt.subplots(figsize=(6, 6))
def cuadro(k):
    ax.clear()
    gr.mapa_campo(ax, ALON, ALAT, UZT[k], AZ, f"{EVENTO} · t = {TIEMPOS[k]:.0f} s", vmax_t, barra=False)
    act, hecha = dfm.frente_de_ruptura(FK, TIEMPOS[k])
    gr.contorno_falla(ax, FK, color="0.6")
    gr.dibujar_falla(ax, FK, campo=np.ones(len(FK)), cmap="Greys", vmax=1.5, alpha=0.45, solo=act)
    gr.marcar(ax, LOC_LON, LOC_LAT, LOCALIDAD, "o", "crimson", 6)
ani = animation.FuncAnimation(fig, cuadro, frames=len(TIEMPOS))
GIF = f"ruptura_{dfm.slug(EVENTO)}.gif"
ani.save(GIF, writer=animation.PillowWriter(fps=6)); plt.close(fig)
display(Image(filename=GIF))

iL = np.unravel_index(np.argmin((ALON - LOC_LON) ** 2 + (ALAT - LOC_LAT) ** 2), ALON.shape)
iM = np.unravel_index(np.argmax(UZT[-1]), ALON.shape)
plt.figure(figsize=(8, 3))
plt.plot(TIEMPOS, UZT[:, iL[0], iL[1]], label=LOCALIDAD, color="crimson")
plt.plot(TIEMPOS, UZT[:, iM[0], iM[1]], label="punto de máximo levantamiento", color="#B04616")
plt.xlabel("tiempo desde el inicio (s)"); plt.ylabel("uz (m)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

📝 **Preguntas (Paso 4)**
1. ¿Hacia dónde se propagó la ruptura (unilateral o bilateral)? ¿Cuánto duró? Compárenlo con su predicción.
2. Un tsunami viaja a c = √(g h). ¿Cuánto tarda en recorrer 50 km sobre 2000 m de agua? Compárenlo con la duración de la ruptura: ¿es razonable tratar la deformación como **instantánea** para el tsunami? ¿Dejaría de serlo en una ruptura de 1000 km?
3. ¿En qué momento empieza a moverse su localidad y cuándo alcanza su desplazamiento final?

---
## Paso 5 · Del fondo marino a la superficie del mar

Lo que levanta el agua no es solo el **uz** de Okada. Se agregan dos efectos:

**1. Aporte del movimiento horizontal** (Tanioka y Satake, 1996). Si el fondo tiene pendiente y se desplaza en horizontal, en un punto fijo el fondo sube o baja, aunque uz sea cero. Imaginen una rampa que se corre hacia el lado profundo: donde antes había agua aparece la parte alta de la rampa.

$$u_h = u_E \, \frac{\partial H}{\partial x_E} + u_N \, \frac{\partial H}{\partial x_N}$$

donde H es la profundidad del agua (positiva). Es importante en **taludes empinados** y con **grandes desplazamientos horizontales** (Tohoku 2011, con decenas de metros de movimiento cerca de la fosa).

**2. Filtro de Kajiura** (1963). El agua no copia los detalles del fondo más pequeños que su profundidad: una columna de 4 km de agua «suaviza» la deformación. En el espacio de números de onda k:

$$\eta_0(k) = \frac{u_{\text{fondo}}(k)}{\cosh(k\, h)}$$

Las formas más angostas que la profundidad (k h grande) se atenúan mucho; las anchas pasan casi iguales. Como h cambia de un punto a otro, el filtro se aplica para varias profundidades y en cada punto se interpola según su profundidad local.

**`FUENTE` ✏️:** elijan qué modelo de falla se usa desde aquí hasta la exportación: `falla finita`, `uniforme` o `falla finita con μ(z)`. Justifiquen la elección en el informe.

**Cómo leer los mapas:** solo se muestra el mar. Izquierda: uz de Okada; centro: el aporte horizontal (con su propia escala, suele ser menor); derecha: **η0**, la superficie inicial del mar, que es la **condición inicial del tsunami**.

> ⚠️ La pendiente sale de ETOPO 2022 a 1′ (unos 1,8 km). Los taludes más abruptos que esa resolución se suavizan y el aporte horizontal puede quedar subestimado.

In [ ]:
#@title ✏️ Fuente para el resto del capítulo
FUENTE = "falla finita"   #@param ["falla finita", "uniforme", "falla finita con μ(z)"]
if FUENTE == "falla finita" and FF is not None:
    F_SEL = FF
elif FUENTE == "falla finita con μ(z)":
    F_SEL = BASE_MU
else:
    F_SEL = FU
    if FUENTE != "uniforme": print("⚠️ No hay falla finita: se usa el rectángulo uniforme.")
UE, UN, UZ = F_SEL.desplazamiento(LON, LAT, NU)
UH = dfm.aporte_horizontal(UE, UN, LON, LAT, Z)
FONDO = UZ + UH                      # deformación efectiva del fondo marino
ETA0 = dfm.filtro_kajiura(FONDO, Z, LON, LAT)

vm = np.abs(FONDO).max()
fig, axs = plt.subplots(1, 3, figsize=(17, 5.6))
gr.mapa_campo(axs[0], LON, LAT, np.where(Z < 0, UZ, np.nan), Z, "uz (Okada) en el mar", vm)
gr.mapa_campo(axs[1], LON, LAT, np.where(Z < 0, UH, np.nan), Z, "aporte horizontal uh", max(np.abs(UH).max(), 1e-3))
gr.mapa_campo(axs[2], LON, LAT, np.where(Z < 0, ETA0, np.nan), Z, "η0: superficie inicial del mar (Kajiura)", vm)
plt.tight_layout(); plt.show()

V_uz = dfm.volumen_desplazado(np.where(Z < 0, UZ, 0), LON, LAT, Z)
V_f = dfm.volumen_desplazado(FONDO, LON, LAT, Z)
E_eta = dfm.energia_potencial(ETA0, LON, LAT, Z)
print(f"Volumen levantado: solo uz {V_uz[0]/1e9:.2f} km³ · con uh {V_f[0]/1e9:.2f} km³ "
      f"({100*(V_f[0]-V_uz[0])/max(V_uz[0],1):+.0f} %)")
print(f"Máximo en el mar: uz {np.where(Z<0,UZ,-9).max():.2f} m · fondo {np.where(Z<0,FONDO,-9).max():.2f} m · η0 {ETA0.max():.2f} m")
print(f"Energía potencial inicial del tsunami ≈ {E_eta:.2e} J")

### 🧭 Bitácora · Decisión 2 ✏️ · ¿Qué fuente entregan a la Unidad 3?
Esta es la decisión más importante del capítulo: la fuente que elijan será la condición inicial del tsunami que modelarán en las Unidades 3 y 4. Comparen las tres opciones (uniforme, falla finita, falla finita con μ(z)) con lo que ya saben de los Pasos 2 y 3 y, cuando lo tengan, con la validación del Paso 6. **Pueden volver a esta celda y cambiarla después del Paso 6**; si cambian, expliquen por qué en la evidencia.

In [ ]:
#@title ✏️ Decisión 2 · Fuente para la Unidad 3
decision(
    paso="5",
    pregunta="¿Qué fuente entrego como condición inicial del tsunami?",
    alternativas="uniforme | falla finita | falla finita con μ(z)",
    eleccion=FUENTE,
    evidencia="...",      # p. ej.: métricas del Paso 2, RMSE del Paso 6, disponibilidad de datos
    riesgo="...",         # p. ej.: "si la falla finita sobreestima el deslizamiento somero, la ola inicial sería ..."
)

**Perfil fondo–superficie.** Arriba: uz (verde azulado), uz + uh (naranja) y η0 (azul oscuro, línea gruesa) por su localidad. Abajo: el fondo marino en el mismo perfil, para ver dónde está el talud. La celda siguiente es una vista 3D interactiva de η0 (pueden girarla con el ratón).

In [ ]:
uzp = dfm.interpolar_malla(LON, LAT, np.where(Z < 0, UZ, 0), P2LON, P2LAT)
fop = dfm.interpolar_malla(LON, LAT, FONDO, P2LON, P2LAT)
etp = dfm.interpolar_malla(LON, LAT, ETA0, P2LON, P2LAT)
zpp = dfm.interpolar_malla(BLON, BLAT, BZ, P2LON, P2LAT)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))
a1.plot(S2, uzp, color="#1C6E7D", label="uz"); a1.plot(S2, fop, color="#B04616", label="uz + uh")
a1.plot(S2, etp, color="#12263A", lw=2.5, label="η0 (Kajiura)"); a1.axhline(0, color="0.4", lw=0.8)
a1.legend(); a1.set_ylabel("m"); a1.set_title(f"Fondo y superficie del mar · perfil por {LOCALIDAD}")
a2.fill_between(S2, zpp, zpp.min() - 200, color="#E4DAC4"); a2.plot(S2, zpp, "k", lw=1)
a2.set_ylabel("elevación (m)"); a2.set_xlabel("km (negativo = mar)")
plt.tight_layout(); plt.show()

In [ ]:
# Vista 3D interactiva (opcional): fondo deformado y superficie inicial del mar
try:
    import plotly.graph_objects as go
    k = max(1, LON.shape[0] // 120)
    fig3 = go.Figure([go.Surface(x=LON[::k, ::k], y=LAT[::k, ::k], z=np.where(Z < 0, ETA0, np.nan)[::k, ::k],
                                 colorscale="RdBu_r", cmid=0, name="η0", colorbar=dict(title="η0 (m)"))])
    fig3.update_layout(title="Superficie inicial del mar (η0)", height=550,
                       scene=dict(xaxis_title="lon", yaxis_title="lat", zaxis_title="m", aspectratio=dict(x=1, y=1, z=0.35)))
    fig3.show()
except ImportError:
    print("plotly no disponible")

📝 **Preguntas (Paso 5)**
1. ¿Cuánto aporta el movimiento horizontal al volumen levantado (en %)? ¿En qué zona: talud, fosa o plataforma?
2. ¿Cuánto reduce el filtro de Kajiura el máximo? ¿En qué eventos esperan que el efecto sea mayor (aguas más profundas, fuentes más pequeñas)?
3. Describan η0: ¿cresta hacia el mar y valle hacia la costa? ¿Qué llegará primero a su localidad, una subida o un retiro del mar?
4. La energía potencial inicial es E = ½ ρ g ∬ η0² dA. Compárenla con la energía sísmica del evento (orden de magnitud) y comenten.

---
## Paso 6 ✏️ · Validación con observaciones

Un modelo vale lo que acierta. Las observaciones de deformación cosísmica son de tres tipos:
- **GPS:** desplazamientos este, norte y vertical en estaciones; los más precisos (milímetros a centímetros).
- **InSAR:** desplazamiento en la línea de vista del satélite, sobre tierra. Para compararlo con este cuaderno hay que convertirlo (o usar los valores verticales que publique el estudio).
- **Costa:** levantamiento o hundimiento medido en campo con marcas biológicas (algas, corales, mejillones), mareógrafos o testigos.

Las observaciones están en `datos/eventos/<evento>/observaciones.csv` con las columnas `nombre, lat, lon, uE_m, uN_m, uz_m, sigma_m, tipo, fuente`. Complétenlas con la **tabla de observaciones del Capítulo 1**, o agréguenlas en `OBS_EXTRA`. Cada observación necesita coordenadas, valor y **fuente citable**. Dejen vacías las componentes que no tengan.

**Qué se calcula:** el residual (observado − modelado) en cada punto y el **RMSE** por componente,

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{k=1}^{n} \left(u_k^{\text{obs}} - u_k^{\text{mod}}\right)^2}$$

para los tres modelos (uniforme, falla finita y falla finita con μ(z)).

**Cómo leer las figuras:** a la izquierda, el mapa de uz del modelo elegido con las observaciones como círculos del mismo código de colores (un círculo «del mismo color que el fondo» es un acierto) y flechas para el movimiento horizontal. A la derecha, el gráfico 1:1: cuanto más cerca de la diagonal, mejor.

In [ ]:
#@title ✏️ Observaciones adicionales
OBS_EXTRA = [
    # dict(nombre="EST1", lat=..., lon=..., uE=None, uN=None, uz=..., sigma=None, tipo="costa", fuente="Autor (año)"),
]
OBS = D["observaciones"] + OBS_EXTRA
if not OBS:
    print("⚠️ Sin observaciones cargadas. Completen observaciones.csv o OBS_EXTRA y vuelvan a ejecutar.")
else:
    res = {}
    for nombre_f, F in (("uniforme", FU), ("falla finita", FF), ("falla finita con μ(z)", BASE_MU if FF is not None else None)):
        if F is None: continue
        filas, rmse = dfm.validar(F, OBS, NU); res[nombre_f] = rmse
    print("RMSE (m) por componente:"); display(pd.DataFrame(res).T.round(3))
    filas, _ = dfm.validar(F_SEL, OBS, NU)
    display(pd.DataFrame(filas).round(3))
    fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
    gr.mapa_campo(axs[0], LON, LAT, UZ, Z, f"Observado (círculos) y modelado · {FUENTE}", np.abs(UZ).max())
    for o in OBS:
        if o["uz"] is not None:
            axs[0].scatter(o["lon"], o["lat"], c=[o["uz"]], cmap="RdBu_r", vmin=-np.abs(UZ).max(), vmax=np.abs(UZ).max(), s=70, edgecolor="k", zorder=6)
        if o["uE"] is not None and o["uN"] is not None:
            axs[0].quiver(o["lon"], o["lat"], o["uE"], o["uN"], color="k", scale=10, zorder=7)
    mod = dfm.validar(F_SEL, OBS, NU)[0]
    obs_z = [f.get("uz_obs") for f in mod]; mod_z = [f["uz_mod"] for f in mod]
    pares = [(a, b) for a, b in zip(obs_z, mod_z) if a is not None]
    if pares:
        a, b = np.array(pares).T; lim = max(np.abs(a).max(), np.abs(b).max()) * 1.1
        axs[1].plot([-lim, lim], [-lim, lim], "k--", lw=0.8); axs[1].scatter(a, b, c="#B04616")
        axs[1].set_xlabel("uz observado (m)"); axs[1].set_ylabel("uz modelado (m)"); axs[1].set_aspect("equal")
    plt.tight_layout(); plt.show()

### 🧭 Bitácora · Decisión 3 ✏️ · ¿Qué observaciones usan y cuánto confían en ellas?
No todas las observaciones valen lo mismo: un GPS con error de milímetros no se compara con una marca de algas medida con decímetros de incertidumbre, y algunos datos incluyen deformación posterior al sismo. Decidan cuáles usan para juzgar el modelo y por qué.

In [ ]:
#@title ✏️ Decisión 3 · Observaciones
decision(
    paso="6",
    pregunta="¿Qué observaciones uso para validar y cuánto confío en ellas?",
    alternativas="...",   # p. ej.: "todas | solo GPS | GPS + costa, sin InSAR"
    eleccion="...",
    evidencia="...",      # tipo, error (sigma), fecha de la medición, fuente
    riesgo="...",         # qué conclusión podría ser falsa por esta elección
)

### 🧭 Diagnóstico ✏️ · ¿Por qué falla el modelo donde falla?

Todo modelo falla en algún lugar. La tarea es **explicar por qué** y **probarlo**, como hace un investigador:

1. **Anomalía.** Describan la mayor diferencia entre su modelo y la realidad. Puede ser un residual grande del Paso 6, un desacuerdo entre la falla finita y sus observaciones, o, si no tienen observaciones, la diferencia entre el tsunami observado de su evento (Capítulo 1) y la η0 de este cuaderno.
2. **Dos hipótesis** que la expliquen. Por ejemplo: deslizamiento somero subestimado, rigidez, geometría plana, centroide desplazado, deformación posterior al sismo en los datos o resolución de la batimetría.
3. **Una prueba** que permita distinguirlas con las herramientas del cuaderno, y su **resultado**.

> Si la falla finita y las observaciones **no concuerdan**, no hay una respuesta correcta predefinida: decidan qué creer y justifíquenlo. Ese razonamiento es lo que se evalúa.

In [ ]:
#@title ✏️ Diagnóstico de la anomalía
DIAGNOSTICO = dict(
    anomalia="...",        # qué difiere, dónde y cuánto (con cifras)
    hipotesis_1="...",
    hipotesis_2="...",
    prueba="...",          # qué cambian en el modelo para distinguirlas (celda siguiente)
    resultado="...",       # qué pasó con el RMSE o con la anomalía
    conclusion="...",      # qué hipótesis queda mejor apoyada y qué no pueden afirmar
)
print("Diagnóstico registrado." if "..." not in DIAGNOSTICO.values() else "⚠️ Completen todos los campos del diagnóstico.")

**Prueba de la hipótesis ✏️.** Modifiquen la fuente según su hipótesis y vuelvan a validar. Algunos ejemplos ya escritos en el paquete:
- rigidez somera distinta: `dfm.reescalar_por_rigidez(BASE, mu_somero=5e9, mu_profundo=40e9, z1=5, z2=20)`;
- un rectángulo más somero o con otro buzamiento: `dfm.falla_uniforme(lon, lat, prof, rumbo, buz, rake, M0=..., L=..., W=...)`;
- más deslizamiento cerca de la fosa: `FF.con_slip(nuevo_slip)` con un arreglo modificado.

In [ ]:
#@title ✏️ Prueba de la hipótesis
F_PRUEBA = dfm.reescalar_por_rigidez(BASE, mu_somero=5e9, mu_profundo=40e9, z1=5, z2=20)   # ✏️ cambien según su hipótesis

if OBS:
    _, rmse_ref = dfm.validar(F_SEL, OBS, NU)
    _, rmse_pru = dfm.validar(F_PRUEBA, OBS, NU)
    display(pd.DataFrame({"modelo elegido": rmse_ref, "prueba": rmse_pru}).T.round(3))
uz_ref = F_SEL.desplazamiento(P2LON, P2LAT, NU)[2]
uz_pru = F_PRUEBA.desplazamiento(P2LON, P2LAT, NU)[2]
plt.figure(figsize=(10, 3.5))
plt.plot(S2, uz_ref, color="#B04616", lw=2, label=f"modelo elegido ({FUENTE})")
plt.plot(S2, uz_pru, color="#12263A", lw=2, ls="--", label="prueba de la hipótesis")
plt.axhline(0, color="0.4", lw=0.8); plt.axvline(0, color="crimson", lw=1)
plt.xlabel("km (negativo = mar)"); plt.ylabel("uz (m)"); plt.legend(); plt.title("Prueba de la hipótesis · perfil por la localidad")
plt.show()

📝 **Preguntas (Paso 6)**
1. ¿Qué modelo tiene menor RMSE? ¿En qué componente falla más?
2. ¿Los errores se concentran en una zona (cerca de la fosa, en la costa, lejos)? ¿Qué supuesto lo explica: semiespacio homogéneo, geometría plana, deslizamiento uniforme, deformación posterior al sismo incluida en los datos?
3. Si su evento no tiene observaciones, expliquen por qué y qué dato permitiría validarlo.

---
## Paso 7 · Sensibilidad: ¿qué parámetro manda?

Se cambia **un parámetro a la vez** del rectángulo uniforme y se mide cuánto cambia el **máximo de η0** (la ola inicial). Al cambiar L o W se recalcula el deslizamiento para **conservar M0**: no se puede agrandar la falla sin repartir el mismo momento.

| Parámetro | Variación |
|---|---|
| Profundidad del centroide | ± 5 km |
| Buzamiento | ± 5° |
| Rake | ± 15° |
| Largo L | ± 20 % (mismo M0) |
| Ancho W | ± 20 % (mismo M0) |

**Cómo leer el gráfico de tornado:** cada fila es un parámetro, ordenadas de mayor a menor efecto. La barra verde azulada es el cambio al **disminuir** el parámetro y la naranja al **aumentarlo**. El parámetro de arriba es el que más controla el resultado y, por lo tanto, el que más conviene conocer bien.

In [ ]:
def eta_max(F, paso=PASO * 2):
    lo, la = dfm.malla(REG, paso); z = dfm.interpolar_malla(BLON, BLAT, BZ, lo, la)
    ue, un, uz = F.desplazamiento(lo, la, NU)
    fo = uz + dfm.aporte_horizontal(ue, un, lo, la, z)
    et = dfm.filtro_kajiura(fo, z, lo, la)
    return float(et.max()), dfm.volumen_desplazado(fo, lo, la, z)[0] / 1e9

from dataclasses import replace
from deformacion.fuentes import Falla
base_e, base_v = eta_max(FU)
VARIACIONES = {"prof. centroide ±5 km": ("prof", 5.0), "buzamiento ±5°": ("buz", 5.0),
               "rake ±15°": ("rake", 15.0), "L ±20 %": ("L", 0.2), "W ±20 %": ("W", 0.2)}
filas = []
for etiqueta, (attr, dv) in VARIACIONES.items():
    vals = []
    for sgn in (-1, 1):
        s = replace(FU.subfallas[0])
        v0 = getattr(s, attr)
        nuevo = v0 * (1 + sgn * dv) if attr in ("L", "W") else v0 + sgn * dv
        setattr(s, attr, nuevo)
        if s.prof_sup < 0.5:       # el borde superior no puede quedar sobre el fondo
            s.prof += 0.5 - s.prof_sup
        if attr in ("L", "W"):     # mismo M0: el slip compensa el área
            s.slip = FU.subfallas[0].slip * (v0 / nuevo)
        vals.append(eta_max(Falla([s]))[0])
    filas.append(dict(parametro=etiqueta, menos=vals[0] - base_e, mas=vals[1] - base_e))
T7 = pd.DataFrame(filas).set_index("parametro")
orden = (T7.abs().max(axis=1)).sort_values().index
fig, ax = plt.subplots(figsize=(8, 3.5))
yy = np.arange(len(orden))
ax.barh(yy - 0.2, T7.loc[orden, "menos"], height=0.4, color="#1C6E7D", label="parámetro −")
ax.barh(yy + 0.2, T7.loc[orden, "mas"], height=0.4, color="#B04616", label="parámetro +")
ax.set_yticks(yy); ax.set_yticklabels(orden)
ax.axvline(0, color="k"); ax.set_xlabel(f"cambio del máximo de η0 (m) · base {base_e:.2f} m"); ax.legend()
ax.set_title("Sensibilidad del rectángulo uniforme (mismo M0)"); plt.tight_layout(); plt.show()

📝 **Preguntas (Paso 7)**
1. ¿Qué parámetro produce el mayor cambio? ¿Coincide con el que identificaron como más incierto en el Capítulo 1?
2. ¿Por qué, al cambiar L o W manteniendo M0, el resultado cambia aunque el sismo «sea el mismo»?
3. ¿Qué dato adicional (réplicas, GPS, falla finita de otra agencia) reduciría más la incertidumbre de su modelo?

---
## Paso 8 · Predicción frente a modelo

Comparen lo que predijeron en el Paso P con lo que calculó el modelo. La pregunta clave para el informe no es cuánto se equivocaron, sino **qué razonamiento falló o acertó**.

In [ ]:
import pandas as pd
_m_ff = dfm.metricas(LON, LAT, U_FF[2], Z) if U_FF is not None else dfm.metricas(LON, LAT, U_UNI[2], Z)
_m_un = dfm.metricas(LON, LAT, U_UNI[2], Z)
_uz_loc = float(dfm.interpolar_malla(LON, LAT, (U_FF if U_FF is not None else U_UNI)[2],
                                     np.array([[LOC_LON]]), np.array([[LOC_LAT]]))[0, 0])
_eta_vs = "mayor" if ETA0.max() > 1.02 * FONDO[Z < 0].max() else ("menor" if ETA0.max() < 0.98 * FONDO[Z < 0].max() else "igual")
COMPARACION = pd.DataFrame([
    dict(magnitud="Levantamiento máximo, falla finita (m)", prediccion=PREDICCION["uz_max_falla_finita_m"], modelo=round(_m_ff["uz_max_m"], 2)),
    dict(magnitud="¿Mayor que el uniforme?", prediccion=PREDICCION["mayor_que_uniforme"], modelo="sí" if _m_ff["uz_max_m"] > _m_un["uz_max_m"] else "no"),
    dict(magnitud="Fracción del levantamiento en el mar", prediccion=PREDICCION["fraccion_en_mar"], modelo=round(_m_ff.get("fraccion_levantamiento_en_mar") or 0, 2)),
    dict(magnitud=f"{LOCALIDAD}", prediccion=PREDICCION["localidad"], modelo=f"{'sube' if _uz_loc > 0 else 'baja'} ({_uz_loc:+.2f} m)"),
    dict(magnitud="Duración de la ruptura (s)", prediccion=PREDICCION["duracion_ruptura_s"], modelo=round(float(TIEMPOS[-1]))),
    dict(magnitud="η0 máximo frente al fondo", prediccion=PREDICCION["eta0_frente_al_fondo"], modelo=f"{_eta_vs} ({ETA0.max():.2f} vs {FONDO[Z < 0].max():.2f} m)"),
])
print("Razonamiento registrado:", PREDICCION["razonamiento"])
COMPARACION

### 🧭 Cierre metacognitivo ✏️
Respondan en dos o tres líneas cada pregunta. No son preguntas de contenido sino **sobre cómo resolvieron el problema**. Se califican por su honestidad y concreción, no por haber hecho todo bien.

In [ ]:
#@title ✏️ Cierre metacognitivo
METACOGNICION = dict(
    estrategia="...",     # ¿Cómo abordaron el problema? ¿Qué hicieron primero y por qué?
    atasco="...",         # ¿En qué punto se trabaron y cómo lo resolvieron (o no)?
    transferencia="...",  # Si mañana les asignan otro evento, ¿qué harían distinto?
)
print("Cierre registrado." if "..." not in METACOGNICION.values() else "⚠️ Completen las tres respuestas.")
mostrar_bitacora()

---
## Paso 9 · Exportación: el paquete para la Unidad 3

En la Unidad 3 se propagará el tsunami con un modelo de aguas someras (GeoClaw u otro). Ese modelo necesita **la deformación del fondo marino** y **la batimetría**. Esta celda los guarda en `datos/eventos/<evento>/modelo_U2/`:

| Archivo | Contenido | Para qué |
|---|---|---|
| `dtopo_estatico.tt3` | Deformación del fondo (uz + uh) | Condición inicial de GeoClaw (dtopo tipo 3) |
| `dtopo_cinematico.tt3` | uz creciendo en el tiempo según la ruptura (sin uh) | Generación con ruptura no instantánea |
| `campos_U2.nc` / `.npz` | uE, uN, uz, uh, fondo, η0 y batimetría en la malla | Análisis y figuras en otros programas (QGIS, Python) |
| `topo_regional.tt3` / `.nc` | Batimetría ETOPO 2022 de la región | Malla de propagación |
| `resumen_U2.json` | Fuente elegida, parámetros, métricas, RMSE y supuestos | Trazabilidad del modelo |
| `ruptura_<evento>.gif` | Animación de la ruptura | Informe y presentación |

**¿dtopo o η0?** GeoClaw recibe la deformación del **fondo** (dtopo) y calcula por su cuenta cómo responde el agua. η0 (con Kajiura) es la superficie inicial ya filtrada: sirve para modelos que arrancan desde una superficie del mar dada y para comparar. Ambos quedan en `campos_U2.nc`.

La última celda descarga un `.zip` con todo: es el entregable del Paso 9 y la entrada de la Unidad 3.

In [ ]:
SALIDA = os.path.join(D["carpeta"], "modelo_U2"); os.makedirs(SALIDA, exist_ok=True)
escribir_dtopo_tt3(os.path.join(SALIDA, "dtopo_estatico.tt3"), LON, LAT, FONDO)
# Versión en el tiempo: solo uz (el aporte horizontal se incluye en la versión estática)
FKIN = F_SEL if len(F_SEL) > 1 else dfm.subdividir(FU, 16, 8, FU.hipocentro, vr=VR)
T_EXP = dfm.tiempos_sugeridos(FKIN, 24)
FONDO_T = dfm.campo_en_el_tiempo(FKIN.desplazamiento_por_subfalla(ALON, ALAT, NU, ("uz",))["uz"], FKIN, T_EXP)
escribir_dtopo_tt3(os.path.join(SALIDA, "dtopo_cinematico.tt3"), ALON, ALAT, FONDO_T, T_EXP)
archivos = guardar_campos(os.path.join(SALIDA, "campos_U2"), LON, LAT,
                          dict(uE=UE, uN=UN, uz=UZ, uh=UH, fondo=FONDO, eta0=ETA0, z_topo=Z),
                          dict(evento=EVENTO, fuente=FUENTE, grupo=GRUPO))
escribir_topo_tt3(os.path.join(SALIDA, "topo_regional.tt3"), BLON, BLAT, BZ)
guardar_campos(os.path.join(SALIDA, "topo_regional"), BLON, BLAT, dict(z=BZ), dict(fuente="ETOPO 2022"))
if os.path.exists(GIF):
    import shutil; shutil.copy(GIF, SALIDA)

RESUMEN = dict(grupo=GRUPO, evento=EVENTO, localidad=dict(nombre=LOCALIDAD, lat=LOC_LAT, lon=LOC_LON),
               fuente_exportada=FUENTE, falla=F_SEL.resumen(), ficha_uniforme=FU.resumen(),
               subfalla_uniforme=vars(FU.subfallas[0]), nu=NU,
               rigidez=dict(mu_somero=MU_SOMERO, mu_profundo=MU_PROFUNDO, z1=Z1, z2=Z2),
               metricas_uz=dfm.metricas(LON, LAT, UZ, Z), eta0_max_m=float(ETA0.max()), eta0_min_m=float(ETA0.min()),
               volumen_levantado_km3=V_f[0] / 1e9, energia_J=E_eta, malla=dict(region=REG, paso_grados=PASO),
               supuestos=["semiespacio elástico homogéneo e isótropo (Okada, 1985)", f"ν = {NU}",
                          "aporte horizontal de Tanioka y Satake (1996)", "filtro de Kajiura (1963) con profundidad local",
                          "batimetría ETOPO 2022"], usgs=META.get("id_usgs"))
RESUMEN.update(bitacora=BITACORA, diagnostico=DIAGNOSTICO, metacognicion=METACOGNICION)
if BITACORA:
    pd.DataFrame(BITACORA).to_csv(os.path.join(SALIDA, "bitacora_decisiones.csv"), index=False)
with open(os.path.join(SALIDA, "resumen_U2.json"), "w", encoding="utf-8") as fh:
    json.dump(RESUMEN, fh, ensure_ascii=False, indent=2, default=float)
print("Guardado en", SALIDA); print(sorted(os.listdir(SALIDA)))

In [ ]:
import shutil
ZIP = shutil.make_archive(f"U2_{dfm.slug(EVENTO)}_{GRUPO.replace(' ', '_')}", "zip", SALIDA)
try:
    from google.colab import files; files.download(ZIP)
except ImportError:
    print("Archivo listo:", ZIP)

---
## Preguntas para el informe (Capítulo 2)

1. **Predicción.** ¿Qué acertaron y qué no en el Paso P? ¿Qué idea previa corrigieron?
2. **Datos.** Describan el modelo de falla finita de su evento (origen, número de subfallas, deslizamiento máximo y medio, profundidad) y sus limitaciones.
3. **Geometría.** ¿Cómo es la interfase en profundidad y qué tan bien la representa el rectángulo de su ficha?
4. **Uniforme frente a finita.** ¿Qué cambia en el levantamiento máximo, el hundimiento, la línea de cero y la fracción en el mar? ¿Por qué?
5. **Rigidez.** ¿Qué efecto tiene una roca somera blanda y qué valores adoptan para su evento? Justifíquenlo.
6. **Cinemática.** Dirección y duración de la ruptura; ¿es válida la aproximación instantánea para su tsunami?
7. **Del fondo al mar.** Aporte horizontal y filtro de Kajiura: cuánto cambian la condición inicial y por qué.
8. **Validación.** RMSE de cada modelo, dónde acierta y dónde falla, y causas probables.
9. **Sensibilidad.** Qué parámetro manda y qué dato reduciría la incertidumbre.
10. **Su localidad.** ¿Sube o baja? ¿Qué llega primero, subida o retiro del mar? Consecuencias para la gestión del riesgo (sin entrar aún en daños: eso corresponde a otras asignaturas).
11. **Pregunta guía de su evento** (aparece al inicio del cuaderno).
12. **Proceso.** Presenten su bitácora de decisiones y su diagnóstico: ¿qué decisión fue la más difícil, con qué evidencia la tomaron y qué riesgo asumen? ¿Qué aprendieron de su forma de resolver el problema?

### ✅ Lista de entrega
- [ ] Predicción registrada y tabla predicción vs. modelo (Pasos P y 8).
- [ ] Mapa de deslizamiento y corte en profundidad con su interpretación (Pasos 0 y 1).
- [ ] Mapas, perfil y tabla de métricas uniforme vs. falla finita (Paso 2).
- [ ] Perfil con μ variable y justificación de la rigidez (Paso 3).
- [ ] Figura de 4 instantes y GIF de la ruptura (Paso 4).
- [ ] Mapas de uz, uh y η0, y perfil fondo–superficie (Paso 5).
- [ ] Validación con al menos 3 observaciones citadas y RMSE (Paso 6).
- [ ] Gráfico de tornado y su lectura (Paso 7).
- [ ] Bitácora con las 3 decisiones, diagnóstico de la anomalía y cierre metacognitivo (🧭).
- [ ] Paquete `modelo_U2` (.zip) entregado (Paso 9); incluye `bitacora_decisiones.csv`.

### Cómo se evalúa el Capítulo 2
Formativa de 2 puntos (avance con retroalimentación) y sumativa de 8 puntos con rúbrica, documento escrito y defensa oral. Las guías completas (en Word, en el aula virtual, y en `guias/Capitulo2_guia_y_rubrica.md`) tienen el detalle. Resumen de la sumativa:

| Criterio | Puntos |
|---|---|
| 1. Respuesta a la retroalimentación (Anexo 2.A) | 0,50 |
| 2. Datos y geometría de la fuente (Pasos 0–1) | 0,75 |
| 3. Uniforme frente a falla finita (Paso 2) | 1,25 |
| 4. Rigidez y cinemática (Pasos 3–4) | 0,75 |
| 5. Del fondo a la superficie y paquete para la U3 (Pasos 5 y 9) | 0,75 |
| 6. Validación y sensibilidad (Pasos 6–7) | 1,00 |
| 7. Discusión, conclusiones y formato | 1,00 |
| **8. Resolución de problemas complejos** (bitácora, diagnóstico y cierre metacognitivo) | **1,00** |
| **9. Defensa oral** (10 min + 5 de preguntas, calificación individual) | **1,00** |
| **Total** | **8,00** |

### Referencias del cuaderno
- Bilek, S. L., y Lay, T. (1999). Rigidity variations with depth along interplate megathrust faults in subduction zones. *Nature, 400*, 443–446.
- Hanks, T. C., y Kanamori, H. (1979). A moment magnitude scale. *Journal of Geophysical Research, 84*(B5), 2348–2350.
- Hayes, G. P., et al. (2018). Slab2, a comprehensive subduction zone geometry model. *Science, 362*(6410), 58–61. https://doi.org/10.1126/science.aat4723
- Kajiura, K. (1963). The leading wave of a tsunami. *Bulletin of the Earthquake Research Institute, 41*, 535–571.
- NOAA NCEI (2022). ETOPO 2022 15 Arc-Second Global Relief Model. https://doi.org/10.25921/fd45-gt74
- Okada, Y. (1985). Surface deformation due to shear and tensile faults in a half-space. *Bulletin of the Seismological Society of America, 75*(4), 1135–1154.
- Strasser, F. O., Arango, M. C., y Bommer, J. J. (2010). Scaling of the source dimensions of interface and intraslab subduction-zone earthquakes with moment magnitude. *Seismological Research Letters, 81*(6), 941–950.
- Tanioka, Y., y Satake, K. (1996). Tsunami generation by horizontal displacement of ocean bottom. *Geophysical Research Letters, 23*(8), 861–864. https://doi.org/10.1029/96GL00736
- Wells, D. L., y Coppersmith, K. J. (1994). New empirical relationships among magnitude, rupture length, rupture width, rupture area, and surface displacement. *Bulletin of the Seismological Society of America, 84*(4), 974–1002.
- U.S. Geological Survey. Earthquake Hazards Program, modelos de falla finita. https://earthquake.usgs.gov/